# Membangun Aplikasi Generasi Gambar (Azure OpenAI)

Ada lebih dari sekadar generasi teks pada LLM. Anda juga bisa menghasilkan gambar dari deskripsi teks. Gambar sebagai modalitas berguna di berbagai bidang seperti MedTech, arsitektur, pariwisata, pengembangan game, pemasaran, dan lainnya. Dalam pelajaran ini kita bekerja dengan model **GPT Image** saat ini di Microsoft Foundry.

## Tujuan pembelajaran

Pada akhir pelajaran ini Anda akan dapat:

- Menjelaskan apa itu generasi gambar dan di mana kegunaannya.
- Memahami keluarga model `gpt-image` dan bagaimana perbedaannya dengan model legacy DALL·E.
- Membangun aplikasi generasi gambar dan mengedit gambar.

## Apa itu generasi gambar?

Model generasi gambar membuat gambar dari sebuah prompt teks. Model modern seperti `gpt-image` mempelajari hubungan antara teks dan gambar selama pelatihan, lalu secara iteratif mengubah noise acak menjadi gambar yang sesuai dengan prompt Anda.

Dua keluarga model gambar yang dikenal luas adalah:

- **`gpt-image` (OpenAI)** — generasi terkini yang digunakan dalam pelajaran ini. Mendukung generasi gambar dari teks dan pengeditan gambar (inpainting dengan masker).
- **Midjourney** — model pihak ketiga populer dengan layanan dan alur kerja berbasis Discord sendiri.

> Model gambar OpenAI yang lebih lama — **DALL·E 2** dan **DALL·E 3** — adalah legacy. DALL·E 3 tidak lagi tersedia untuk deployment baru, dan fitur seperti `create_variation` hanya ada di DALL·E 2. Gunakan model `gpt-image` untuk aplikasi baru.

Di Microsoft Foundry, **`gpt-image-2`** adalah model gambar terbaru dan paling canggih serta direkomendasikan sebagai default. `gpt-image-1.5` dan `gpt-image-1-mini` juga tersedia secara umum.

> **Penting:** model `gpt-image` mengembalikan gambar yang dihasilkan sebagai **base64** (`b64_json`), bukan sebagai URL. Kode Anda mendekode string base64 ke bytes dan menyimpannya — tidak ada URL gambar untuk diunduh.


## Membangun aplikasi generasi gambar pertama Anda

Jadi, apa yang diperlukan untuk membangun aplikasi generasi gambar? Anda memerlukan perpustakaan berikut:

- **python-dotenv**, sangat disarankan untuk menggunakan perpustakaan ini untuk menyimpan rahasia Anda dalam file *.env* yang terpisah dari kode.
- **openai**, perpustakaan ini yang akan Anda gunakan untuk berinteraksi dengan API OpenAI.
- **pillow**, untuk bekerja dengan gambar di Python.

Jika belum dilakukan, ikuti instruksi di halaman [Microsoft Learn](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/create-resource?pivots=web-portal&WT.mc_id=academic-105485-koreyst) untuk membuat sumber daya dan model Microsoft Foundry. Pilih **gpt-image-2** sebagai model (model gambar Azure OpenAI terbaru; DALL·E adalah model lama).

1. Buat file *.env* dengan konten berikut:

    ```text
    AZURE_OPENAI_ENDPOINT=<your endpoint>
    AZURE_OPENAI_API_KEY=<your key>
    AZURE_OPENAI_DEPLOYMENT="gpt-image-2"
    ```

    Cari informasi ini di portal Microsoft Foundry untuk sumber daya Anda di bagian "Deployments".



1. Kumpulkan pustaka di atas dalam sebuah file bernama *requirements.txt* seperti berikut:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Selanjutnya, buat lingkungan virtual dan instal pustaka tersebut:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Untuk Windows, gunakan perintah berikut untuk membuat dan mengaktifkan lingkungan virtual Anda:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Tambahkan kode berikut ke dalam file yang disebut *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError

    # import dotenv
    dotenv.load_dotenv()

    # konfigurasikan klien Azure OpenAI (Microsoft Foundry).
    # Model gambar memerlukan versi API terbaru — periksa dokumen Foundry untuk versi yang dibutuhkan model Anda.
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )

    try:
        # Buat gambar menggunakan API pembuatan gambar
        generation_response = client.images.generate(
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Masukkan teks prompt Anda di sini
            size='1024x1024',
            n=1,
            model=os.environ['AZURE_OPENAI_DEPLOYMENT']   # misalnya "gpt-image-2"
        )
        # Tetapkan direktori untuk gambar yang disimpan
        image_dir = os.path.join(os.curdir, 'images')

        # Jika direktori tidak ada, buatlah
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Inisialisasi path gambar (perhatikan tipe file harus png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # model gpt-image mengembalikan gambar sebagai base64 (b64_json), bukan URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Tampilkan gambar di penampil gambar default
        image = Image.open(image_path)
        image.show()

    # tangkap pengecualian
    except BadRequestError as err:
        print(err)

    ```

Mari kita jelaskan kode ini:

- Pertama, kita mengimpor perpustakaan yang kita butuhkan, termasuk perpustakaan OpenAI, perpustakaan dotenv, modul base64, dan perpustakaan Pillow.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError
    ```

- Selanjutnya, kita memuat variabel lingkungan dari file *.env*.

    ```python
    # impor dotenv
    dotenv.load_dotenv()
    ```

- Setelah itu, kita mengonfigurasi klien Azure OpenAI (Microsoft Foundry).

    ```python
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )
    ```

- Berikutnya, kita menghasilkan gambar:

    ```python
    generation_response = client.images.generate(
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Masukkan teks prompt Anda di sini
        size='1024x1024',
        n=1,
        model=os.environ['AZURE_OPENAI_DEPLOYMENT']
    )
    ```

    Model `gpt-image` mengembalikan gambar sebagai string **base64** dalam `data[0].b64_json`. Kita mendekodenya menjadi byte dan menulisnya ke file — tidak ada URL untuk mengunduh.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Terakhir, kita membuka gambar dan menggunakan penampil gambar standar untuk menampilkannya:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Detail lebih lanjut tentang pembuatan gambar

Mari kita lihat parameter dari `images.generate`:

- **prompt**, adalah teks masuk yang digunakan untuk menghasilkan gambar. Di sini adalah "Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils".
- **size**, adalah ukuran gambar yang dihasilkan (misalnya `1024x1024`, `1536x1024`, `1024x1536`, atau `"auto"`).
- **n**, adalah jumlah gambar yang dihasilkan. Di sini kita menghasilkan satu.
- **model**, adalah nama penyebaran model gambar Anda (misalnya `gpt-image-2`).

> Model gambar tidak menerima parameter `temperature` — itu adalah kontrol generasi teks. Untuk mendapatkan variasi, panggil API lagi; untuk mengurangi variasi, buat prompt Anda lebih spesifik.

## Kemampuan tambahan dalam pembuatan gambar

Anda telah melihat cara menghasilkan gambar dengan beberapa baris kode Python. Model `gpt-image` juga dapat **mengedit** gambar yang sudah ada. Dengan menyediakan gambar, **masker** opsional (yang menandai area yang akan diubah), dan prompt, Anda dapat mengubah sebagian gambar — misalnya, menambahkan topi pada kelinci kita.

```python
response = client.images.edit(
  model=os.environ['AZURE_OPENAI_DEPLOYMENT'],
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# suntingan juga dikembalikan sebagai base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

Gambar dasar hanya berisi kelinci; gambar akhir memiliki topi pada kelinci tersebut.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:
Dokumen ini telah diterjemahkan menggunakan layanan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Meskipun kami berupaya untuk mencapai akurasi, harap diketahui bahwa terjemahan otomatis mungkin mengandung kesalahan atau ketidakakuratan. Dokumen asli dalam bahasa aslinya harus dianggap sebagai sumber yang sah. Untuk informasi penting, disarankan menggunakan terjemahan profesional oleh manusia. Kami tidak bertanggung jawab atas kesalahpahaman atau penafsiran yang keliru yang timbul dari penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
